In [ ]:
repository_filter: list[str] = []
show_context_packages: str = "true"

In [ ]:
from code_data_science import data_table as dt
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
import networkx as nx
import warnings

warnings.simplefilter("ignore")

show_context = show_context_packages.lower() == "true"

df = dt.read_csv("../samples/package_quality_metrics.csv")
df = qu.filter_repos(df, repository_filter)

In [ ]:
cycle_df = df[df["inCycle"].astype(str).str.lower() == "true"].copy()

if len(cycle_df) == 0:
    fig = qu.empty_figure("No dependency cycles found.")
else:
    G = nx.DiGraph()

    # Build nodes and edges from cycle data
    cycle_packages = set(cycle_df["packageName"].tolist())
    pkg_metrics = {}
    for _, row in cycle_df.iterrows():
        pkg = row["packageName"]
        coupling = int(row["afferentCoupling"]) + int(row["efferentCoupling"])
        pkg_metrics[pkg] = {
            "afferent": int(row["afferentCoupling"]),
            "efferent": int(row["efferentCoupling"]),
            "coupling": coupling,
            "instability": float(row["instability"]),
        }
        G.add_node(pkg)

        # Parse cycleMembers and add edges
        members_raw = str(row.get("cycleMembers", ""))
        if members_raw and members_raw != "nan":
            members = [m.strip() for m in members_raw.split(",") if m.strip()]
            for member in members:
                G.add_node(member)
                G.add_edge(pkg, member)
                if member not in pkg_metrics:
                    pkg_metrics[member] = {
                        "afferent": 0, "efferent": 0,
                        "coupling": 0, "instability": 0,
                    }

    # Spring layout
    pos = nx.spring_layout(G, seed=42, k=2.0 / max(1, len(G.nodes()) ** 0.5))

    # Identify cycle edges vs context edges
    cycle_edge_x, cycle_edge_y = [], []
    for u, v in G.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        cycle_edge_x += [x0, x1, None]
        cycle_edge_y += [y0, y1, None]

    # Edge trace (red for cycles)
    edge_trace = go.Scatter(
        x=cycle_edge_x, y=cycle_edge_y,
        mode="lines",
        line=dict(width=1.5, color="#EF5350"),
        hoverinfo="none",
        name="Cycle edge",
    )

    # Node traces
    node_x, node_y, node_text, node_sizes, node_colors = [], [], [], [], []
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        m = pkg_metrics.get(node, {"afferent": 0, "efferent": 0, "coupling": 0, "instability": 0})
        node_sizes.append(max(15, min(50, 10 + m["coupling"] * 3)))
        node_colors.append(m["instability"])
        short = node.split(".")[-1] if "." in node else node
        node_text.append(
            f"<b>{node}</b><br>"
            f"Afferent: {m['afferent']}<br>"
            f"Efferent: {m['efferent']}<br>"
            f"Total coupling: {m['coupling']}<br>"
            f"Instability: {m['instability']:.2f}"
        )

    # Determine which nodes are in the original cycle data vs context
    node_symbols = [
        "circle" if n in cycle_packages else "diamond"
        for n in G.nodes()
    ]

    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode="markers+text",
        marker=dict(
            size=node_sizes,
            color=node_colors,
            colorscale=qu.HEALTH_COLORSCALE[::-1],
            cmin=0, cmax=1,
            colorbar=dict(title="Instability"),
            symbol=node_symbols,
            line=dict(width=1, color="#333"),
        ),
        text=[n.split(".")[-1] for n in G.nodes()],
        textposition="top center",
        textfont=dict(size=9),
        hovertext=node_text,
        hoverinfo="text",
        name="Package",
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        title=f"Dependency Cycle Network ({len(cycle_packages)} packages in cycles)",
        showlegend=False,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        height=700,
        width=900,
        plot_bgcolor="white",
        margin=dict(l=20, r=20, t=60, b=20),
        hovermode="closest",
    )
fig.show()